In [2]:
# M6 - Knapsack Version 1\n# Experimental optimization-style team selection notebook.\n\nimport os
import ast
import random
import pandas as pd
import os

import M3  # we reuse apply_ultra_metric


INPUT_DIR = "../data/v1_input_files/"

# M3 outputs (we only read skills from here)
M3_DIR = "../data/v1_output_teaming/teaming_1698proposals_316researchers/data_uc1_m3/"

# Knapsack outputs 
OUT_BASE = "../data/v1_output_teaming/teaming_1698proposals_316researchers/"
OUT_DIR = OUT_BASE + "data_uc1_m6_legacy/"
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_SIZE = 5      # total team size (target + 3 co-members)
NUM_TEAMS = 10     # number of teams per (proposal, researcher)

print("Writing knapsack outputs to:", OUT_DIR)


Writing knapsack outputs to: ../data/v1_output_teaming/teaming_1698proposals_316researchers/data_uc1_m6_legacy/


## 2. Load proposal metadata + M3 skill files

In [3]:
# Basic proposal info (for id/year/title later)
proposal_info = pd.read_csv(INPUT_DIR + "v1_proposal_links_title_synopsis.csv")

# Using M3 skills directly
df_res_skills = pd.read_csv(M3_DIR + "m3_researcher_skills.csv")
df_prop_skills = pd.read_csv(M3_DIR + "m3_proposal_skills.csv")

print(df_res_skills.shape, df_prop_skills.shape)
df_res_skills.head(2), df_prop_skills.head(2)


(202, 3) (434, 3)


(   Unnamed: 0      researcher_name  \
 0           0  Agostinelli, Forest   
 1           1      Ahmad, Iftikhar   
 
                                               skills  
 0  {'reinforcement learning', 'artificial intelli...  
 1  {'electronic technology', 'tech universityrese...  ,
    Unnamed: 0                              nsf_proposal_links_v1  \
 0           0  https://www.nsf.gov/pubs/2021/nsf21598/nsf2159...   
 1           1  https://www.nsf.gov/pubs/2021/nsf21527/nsf2152...   
 
                                               skills  
 0  {'higher education', 'development', 'material'...  
 1  {'chemical', 'environmental engineering', 'sci...  )

## 3. Convert M3 skill CSVs → dictionaries

In [4]:
# researcher_name -> set(skills)
researcher_skills = {}
for _, row in df_res_skills.iterrows():
    name = row["researcher_name"]
    skills = ast.literal_eval(row["skills"])    # M3 stored as string(set(...))
    researcher_skills[name] = set(skills)

# proposal_link -> set(skills)
proposal_skills = {}
for _, row in df_prop_skills.iterrows():
    link = row["nsf_proposal_links_v1"]
    skills = ast.literal_eval(row["skills"])
    proposal_skills[link] = set(skills)

print("Example researcher:", list(researcher_skills.items())[:1])
print("Example proposal:", list(proposal_skills.items())[:1])


Example researcher: [('Agostinelli, Forest', {'search', 'deep learning', 'bioinformatics', 'reinforcement learning', 'artificial intelligence'})]
Example proposal: [('https://www.nsf.gov/pubs/2021/nsf21598/nsf21598.htm', {'higher education', 'professional development', 'industry', 'economic development', 'material', 'engineering', 'science engineering', 'development'})]


## 4. Value function v(r, p) for knapsack



In [5]:
def researcher_value_for_proposal(r_name, p_link):
    """
    v(r,p) = |S_r ∩ S_p| / |S_p|
    Fraction of proposal skills covered by researcher r.
    """
    S_p = proposal_skills.get(p_link, set())
    S_r = researcher_skills.get(r_name, set())
    if not S_p:
        return 0.0
    overlap = len(S_p & S_r)
    return overlap / len(S_p)


## 5. Generate multiple knapsack teams per pair → k_teaming.csv

Here we:

For each proposal:

compute v(r,p) for all researchers

sort researchers by v(r,p)

For each target researcher:

build up to NUM_TEAMS teams

Team 1 = strict top-K co-members (true knapsack optimum)

Remaining teams = random variations sampled from top candidates (still high-value, but diverse)

ensure no duplicate teams for that pair

In [6]:
import pandas as pd
import random

# Assuming researcher_skills and proposal_skills are dicts: {id: [skill1, skill2]}
k_teaming_rows = []

for p_link in proposal_skills:
    required_skills = set(proposal_skills[p_link])
    
    for target_r in researcher_skills:
        teams_for_pair = []
        
        # --- Team 1: Strict Implementation of Algorithm 1 ---
        team = [target_r]
        # Initial coverage from the target researcher
        covered = set(researcher_skills[target_r]).intersection(required_skills)
        
        while len(team) < TEAM_SIZE:
            best_r = None
            max_new_skills = 0
            
            # Find the researcher with the highest marginal gain
            candidates = [r for r in researcher_skills if r not in team]
            for r in candidates:
                r_skills = set(researcher_skills[r])
                # Gain = skills r has that are required but NOT yet covered
                gain = len(r_skills.intersection(required_skills) - covered)
                
                if gain > max_new_skills:
                    max_new_skills = gain
                    best_r = r
            
            if best_r:
                team.append(best_r)
                # Update covered skills
                new_skills = set(researcher_skills[best_r]).intersection(required_skills)
                covered.update(new_skills)
            else:
                break # No more researchers provide new skills
        
        base_team = team
        teams_for_pair.append(base_team)

        # --- Additional Teams: Stochastic Greedy Variations ---
        # To generate NUM_TEAMS, we can add a bit of randomness to the greedy choice
        attempts = 0
        while len(teams_for_pair) < NUM_TEAMS and attempts < 50:
            attempts += 1
            variant_team = [target_r]
            var_covered = set(researcher_skills[target_r]).intersection(required_skills)
            
            while len(variant_team) < TEAM_SIZE:
                # Get all researchers who provide at least 1 new skill
                potential_gainers = []
                for r in researcher_skills:
                    if r not in variant_team:
                        gain = len(set(researcher_skills[r]).intersection(required_skills) - var_covered)
                        if gain > 0:
                            potential_gainers.append((r, gain))
                
                if not potential_gainers:
                    break
                
                # Instead of always picking the absolute best, pick from top 3 or weighted
                potential_gainers.sort(key=lambda x: x[1], reverse=True)
                # Pick one of the top candidates to create a "variation"
                top_slice = potential_gainers[:3]
                chosen_r = random.choice(top_slice)[0]
                
                variant_team.append(chosen_r)
                var_covered.update(set(researcher_skills[chosen_r]).intersection(required_skills))
            
            if variant_team not in teams_for_pair:
                teams_for_pair.append(variant_team)

        # Padding if necessary
        while len(teams_for_pair) < NUM_TEAMS:
            teams_for_pair.append(base_team)

        for t in teams_for_pair:
            k_teaming_rows.append([p_link, target_r, t])

# Convert to DataFrame
df_k_teaming = pd.DataFrame(k_teaming_rows, columns=["nsf_proposal_links_v1", "researcher", "team"])

## 6. ULTRA scoring via M3.apply_ultra_metric → k_goodness_scores.csv

In [7]:
k_goodness_rows = []

for _, row in df_k_teaming.iterrows():
    p_link = row["nsf_proposal_links_v1"]
    target_r = row["researcher"]
    team = row["team"]                 # list in memory

    team_members = list(set(team))     # ensure uniqueness
    p_skills = proposal_skills.get(p_link, set())

    # researcher_skills is the global M3-based mapping
    goodness = M3.apply_ultra_metric(
        p_skills,
        team_members,
        researcher_skills
    )

    k_goodness_rows.append([p_link, target_r, goodness])

df_k_good = pd.DataFrame(
    k_goodness_rows,
    columns=["nsf_proposal_links_v1", "researcher_name", "goodness"]
)

df_k_good.to_csv(OUT_DIR + "k_goodness_scores.csv", index=False)
df_k_good.head()


,nsf_proposal_links_v1,researcher_name,goodness
0,https://www.nsf.gov/pubs/2021/nsf21598/nsf2159...,"Agostinelli, Forest",0.359375
1,https://www.nsf.gov/pubs/2021/nsf21598/nsf2159...,"Agostinelli, Forest",0.312500
2,https://www.nsf.gov/pubs/2021/nsf21598/nsf2159...,"Agostinelli, Forest",0.359375
3,https://www.nsf.gov/pubs/2021/nsf21598/nsf2159...,"Agostinelli, Forest",0.359375
4,https://www.nsf.gov/pubs/2021/nsf21598/nsf2159...,"Agostinelli, Forest",0.312500


## 7. Aggregate per pair → teaming_uc1_knapsack.csv

Now we are doing what M3 does in its final step:
for each (proposal, researcher):

collect the 10 teams,

collect the 10 goodness scores,

sort by goodness descending,

attach proposal metadata.

In [8]:
# Build link -> (id, year, title) mapping from proposal_info
link_to_meta = {}
for _, row in proposal_info.iterrows():
    link = row["nsf_proposal_links_v1"]
    title = row["title"]

    # Try to parse year/id from URL; if fails, leave blank
    try:
        parts = link.split("/")
        year = parts[4]
        prop_id = parts[5]
    except Exception:
        year = ""
        prop_id = link

    link_to_meta[link] = (prop_id, year, title)

final_rows = []

group_teams = df_k_teaming.groupby(["nsf_proposal_links_v1", "researcher"])
group_good = df_k_good.groupby(["nsf_proposal_links_v1", "researcher_name"])

for (p_link, r_name), team_group in group_teams:
    teams_list = team_group["team"].tolist()  # ~10 teams

    try:
        goodness_list = group_good.get_group((p_link, r_name))["goodness"].tolist()
    except KeyError:
        continue

    # sort by goodness desc, zip/unzip
    pairs = sorted(zip(goodness_list, teams_list), reverse=True, key=lambda x: x[0])
    sorted_goodness = [g for g, _ in pairs]
    sorted_teams = [t for _, t in pairs]

    prop_id, year, title = link_to_meta.get(p_link, ("", "", ""))
    skills = proposal_skills.get(p_link, set())

    final_rows.append([
        prop_id,
        year,
        p_link,
        title,
        skills,
        r_name,
        sorted_teams,
        sorted_goodness
    ])

df_final = pd.DataFrame(
    final_rows,
    columns=[
        "proposal_id", "year", "proposal_link", "title",
        "skills", "researcher_name", "team", "goodness"
    ]
)

df_final.to_csv(OUT_BASE + "teaming_uc1_m6_legacy_greedy_coverage.csv", index=False)
df_final.head()


,proposal_id,year,proposal_link,title,skills,researcher_name,team,goodness
0,nsf06504,2006,http://www.nsf.gov/pubs/2006/nsf06504/nsf06504...,"George E. Brown, Jr. Network for Earthquake En...","{nsf funding, engineering, earthquake engineer...","Agostinelli, Forest","[[Agostinelli, Forest, Caicedo, Juan, Ali, Moh...","[0.41447368421052627, 0.41447368421052627, 0.4..."
1,nsf06504,2006,http://www.nsf.gov/pubs/2006/nsf06504/nsf06504...,"George E. Brown, Jr. Network for Earthquake En...","{nsf funding, engineering, earthquake engineer...","Ahmad, Iftikhar","[[Ahmad, Iftikhar, Caicedo, Juan, Ali, Mohammo...","[0.41447368421052627, 0.41447368421052627, 0.4..."
2,nsf06504,2006,http://www.nsf.gov/pubs/2006/nsf06504/nsf06504...,"George E. Brown, Jr. Network for Earthquake En...","{nsf funding, engineering, earthquake engineer...","Alexeev, Oleg S.","[[Alexeev, Oleg S., Caicedo, Juan, Ali, Mohamm...","[0.41447368421052627, 0.41447368421052627, 0.4..."
3,nsf06504,2006,http://www.nsf.gov/pubs/2006/nsf06504/nsf06504...,"George E. Brown, Jr. Network for Earthquake En...","{nsf funding, engineering, earthquake engineer...","Ali, Mohammod","[[Ali, Mohammod, Caicedo, Juan, Gassman, Sarah...","[0.43421052631578944, 0.43421052631578944, 0.4..."
4,nsf06504,2006,http://www.nsf.gov/pubs/2006/nsf06504/nsf06504...,"George E. Brown, Jr. Network for Earthquake En...","{nsf funding, engineering, earthquake engineer...","Ammal, Salai C.","[[Ammal, Salai C., Caicedo, Juan, Ali, Mohammo...","[0.41447368421052627, 0.41447368421052627, 0.4..."


In [9]:
# ==========================================
# 7. GENERATE SUMMARY STATISTICS (Table 3.6 Format)
# ==========================================
import numpy as np

# 1. Calculate the Volume (#T) for each row
df_final['team_volume'] = df_final['team'].apply(len)

# 2. Flatten all goodness scores to get a global Mean and STD
all_scores = [score for sublist in df_final['goodness'] for score in sublist]
mean_g = np.mean(all_scores)
std_g = np.std(all_scores)

# 3. Calculate Average Volume
avg_volume = df_final['team_volume'].mean()

print("--- Method M1 Results (Table 3.6 Style) ---")
print(f"Average Goodness (G): {mean_g:.4f} ± {std_g:.4f}")
print(f"Average Volume (#T):  {int(round(avg_volume))}")

--- Method M1 Results (Table 3.6 Style) ---
Average Goodness (G): 0.3954 ± 0.1132
Average Volume (#T):  10
